# Fine-tuning Qwen3.5-0.8B — hypotheses & ablations

The Part-2 base chosen from the full GPU comparison:
* **qwen3_5-0.8b** — strongest *genuinely sub-1B* (text + multilingual/CJK + chart).

This notebook runs **(0) the A0 prerequisite** — memorization-vs-understanding as a function of
training-data size, which fixes the synthetic data scale — then **(1)** visualises the **baseline
capability gaps**, **(2)** states the **hypothesis** to close each gap (which module to adapt → which
ablation arm), and **(3)** runs an **ablation study per section** with a **before/after** bar.
Training/eval cells need a GPU; run them with `scripts/run_ablation.py`, which writes
`docs/results/ablation_results.json` that the plots below read. Run order: Runtime → Run all.

In [ ]:
# --- install docvlm_eval + GUARANTEE transformers>=5 (Qwen3.5-VL needs it) ---
import os, sys, subprocess, importlib
from pathlib import Path

def _repo_root():
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / "pyproject.toml").exists():
            return c
    return None

root = _repo_root()
if root is None:                       # fresh env (Colab/Kaggle): clone the repo
    subprocess.run(["git", "clone", "https://github.com/SangbumChoi/OCR.git"], check=False)
    root = Path("OCR")
    subprocess.run(["git", "-C", str(root), "checkout", "claude/new-session-w79q0i"], check=False)
    subprocess.run(["git", "-C", str(root), "pull", "--ff-only"], check=False)
os.chdir(root)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[newvlms,finetune,synth]"], check=True)
# belt-and-suspenders: force transformers>=5 (the <5 sweep pin / Colab's preinstalled 4.x would
# otherwise trigger "model type `qwen3_5` ... not recognized").
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=5", "accelerate"], check=True)

# If a stale transformers (<5) is ALREADY imported in this kernel, the on-disk upgrade can't take
# effect until the runtime restarts (common after running the <5 sweep notebook first).
if "transformers" in sys.modules:
    import transformers as _tf
    if int(_tf.__version__.split(".")[0]) < 5:
        print(f"transformers {_tf.__version__} is loaded in this kernel; upgraded on disk -> "
              f"RESTARTING runtime. Re-run this cell after it restarts.")
        os._exit(0)                    # Colab auto-restarts the kernel

src = str(Path.cwd() / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()
import transformers, docvlm_eval
print("transformers", transformers.__version__, "| docvlm_eval", docvlm_eval.__file__)
assert int(transformers.__version__.split(".")[0]) >= 5, \
    "need transformers>=5 for Qwen3.5-VL (got %s)" % transformers.__version__

## 0. Prerequisite (A0) — memorization vs understanding: what synthetic size to use?

The generator can make **infinite** labelled documents, so before any capability ablation we must
answer: as we grow the training set, does the model **understand** the task or just **memorize**
templates? — and therefore **how many synthetic images are worth generating**.

**Protocol (control = only the data size changes).** For each scale *N* (variants/case × 14 cases),
LoRA-train Qwen3.5 for a *fixed* number of epochs on `seed=7` data, then score:
- the **train set** it just fit → *memorization* signal, and
- a **held-out** set on a **different seed (999)**, identical for every *N* → *understanding* signal.

**Read-off.** A large, growing **train − held-out gap** at small *N* = memorization. The size where
the **held-out curve plateaus** is the recommended synthetic scale — generating past it mostly buys
memorization (train → 1.0, held-out flat), not generalization. This A0 result **fixes the data scale
used by A1–A7 below** (`docs/report/ablation_plan.md`, `configs/ablations.yaml` A0_generalization).

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT/"scripts").exists() and (ROOT.parent/"scripts").exists():
    ROOT = ROOT.parent
A0_MODELS = ["qwen3_5-0.8b"]
A0_COLOR  = {"qwen3_5-0.8b": "#d7791d"}
RESULTS = ROOT / "docs" / "results" / "ablation_results.json"

# Run this on the GPU FIRST — it fills models[*]["A0"] that the curve below reads.
print("GPU command (run this first):")
print("  !python scripts/run_ablation.py --arm A0 --a0-sizes 25 50 100 200 --a0-epochs 3")
print("  full curve (heavier): --a0-sizes 50 200 800 3200   | held-out seed 999, FIXED test set\n")

doc = json.loads(RESULTS.read_text()) if RESULTS.exists() else {"models": {}}
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4)); any_data = False
for m in A0_MODELS:
    a0 = doc.get("models", {}).get(m, {}).get("A0")
    if not a0:
        continue
    any_data = True
    sizes = sorted(a0["sizes"], key=lambda s: int(s))
    xs = [a0["sizes"][s]["n_train_samples"] for s in sizes]
    tr = [a0["sizes"][s]["train"].get("score") for s in sizes]
    ho = [a0["sizes"][s]["heldout"].get("score") for s in sizes]
    ax[0].plot(xs, tr, "--o", color=A0_COLOR[m], label=f"{m} train (memorization)")
    ax[0].plot(xs, ho, "-o",  color=A0_COLOR[m], label=f"{m} held-out (understanding)")
    gap = [(t - h) if (t is not None and h is not None) else None for t, h in zip(tr, ho)]
    ax[1].plot(xs, gap, "-o", color=A0_COLOR[m], label=m)
    # recommended size = smallest scale whose held-out is within 3% of the best held-out (plateau)
    hov = [(x, h) for x, h in zip(xs, ho) if h is not None]
    if hov:
        best = max(h for _, h in hov)
        rec = next((x for x, h in hov if h >= 0.97 * best), hov[-1][0])
        print(f"{m}: best held-out={best:.3f} ; recommended synthetic size ~= {rec} train-samples "
              f"(held-out plateau). Past it, train->1.0 while held-out stays flat = memorising.")
ax[0].set_xscale("log"); ax[0].set_xlabel("#training samples (log)"); ax[0].set_ylabel("probe score")
ax[0].set_ylim(0, 1.05); ax[0].set_title("A0 learning curve — train vs held-out")
ax[1].set_xscale("log"); ax[1].set_xlabel("#training samples (log)"); ax[1].set_ylabel("train − held-out")
ax[1].set_title("Memorization gap (up = memorising, not understanding)"); ax[1].axhline(0, color="#999", lw=0.8)
if any_data:
    ax[0].legend(fontsize=8); ax[1].legend(fontsize=8)
else:
    for a in ax:
        a.text(0.5, 0.5, "run the GPU command above to populate", ha="center", va="center", transform=a.transAxes)
plt.tight_layout(); plt.show()

## 1. Baseline capability gaps (from the full GPU run)

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT/"scripts").exists() and (ROOT.parent/"scripts").exists():
    ROOT = ROOT.parent
MODELS = ["qwen3_5-0.8b"]   # single Part-2 base
COLOR = {"qwen3_5-0.8b": "#d7791d"}

# measured baseline from notebooks/colab_full_comparison.ipynb (directional: 1 sample/axis for the
# capability cells; spatial = control-robust PASS; custom-eval/text = multi-sample).
BASE = {
  "qwen3_5-0.8b":   {"T1":1.0,"T2":1.0,"H1":1.0,"H2":0.0,"H3":1.0,"L1":0.0,
                     "spatial_pass":3, "text":0.777, "spot_iou":0.008, "rot180":0.714, "lat_s":13.9},
}
AXES = ["T1","T2","H1","H2","H3","L1"]
_off = lambda i: (i - (len(MODELS)-1)/2)   # centre the bar group for any model count

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
x = np.arange(len(AXES)); w = 0.6
for i, m in enumerate(MODELS):
    ax[0].bar(x + _off(i)*w, [BASE[m][a] for a in AXES], w, label=m, color=COLOR[m])
ax[0].set_xticks(x); ax[0].set_xticklabels(AXES); ax[0].set_ylim(0,1.05)
ax[0].set_title("Capability probe (T·text / H·reasoning / L·location)"); ax[0].legend(fontsize=8)
ax[0].axvspan(2.5, 3.5, color="red", alpha=0.07); ax[0].axvspan(4.5, 5.5, color="red", alpha=0.07)

extra = ["spatial_pass/7","text","spot_iou","rot180"]
vals = lambda m: [BASE[m]["spatial_pass"]/7, BASE[m]["text"], BASE[m]["spot_iou"], BASE[m]["rot180"]]
x2 = np.arange(len(extra))
for i, m in enumerate(MODELS):
    ax[1].bar(x2 + _off(i)*w, vals(m), w, label=m, color=COLOR[m])
ax[1].set_xticks(x2); ax[1].set_xticklabels(extra, fontsize=8); ax[1].set_ylim(0,1.05)
ax[1].set_title("Context-robust + custom-eval signals")
plt.tight_layout(); plt.show()

print("qwen3.5 lacks -> L1 grounding ~0 ; L4 box-tracking = 0 ; H2 relational-compare = 0 ;")
print("                 180-deg rotation retention 0.71 ; latency ~14s/sample")
try:
    import torch
    print("CUDA available:", torch.cuda.is_available(), "| GPU:", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))
except Exception as e:
    print("torch not imported yet (install cell installs .[newvlms]):", e)

## 2. Hypotheses — which module to adapt to close each gap

Capability is **module-localised** (see `docs/report/research_novelty.md`), so each Qwen3.5 gap is
attacked where it physically lives in the encoder→connector→LLM stack:

| Gap | Root module to adapt | Ablation arm | Why | Expected effect |
| --- | --- | --- | --- | --- |
| **L1 grounding / spotting** | vision + **connector** | A1 + A5(connector) | "where" is geometric; the connector serialises positions | spot-IoU ↑, `cap_ground` ↑ |
| **L4 box-tracking** | vision + connector | A1 (sequential boxes) | tracking = repeated localisation | L4 PASS emerges |
| **H2 relational reasoning** | **LLM** attn + mlp | A2 + A5(llm) | multi-region compare is an LM computation | `cap_integ_rel` ↑ |
| **180° rotation** | vision + input | A7 (orientation aug) | legibility/orientation is an encoder property | rot-180 retention ↑ |
| **latency** | — (decode) | A6 (shorter targets / r) | fewer tokens, smaller rank | latency ↓ |

The supervision for A1/A2/A4/A7 is exactly the model-free GT the generator now emits
(`ask_where/region/count/aggregate` + rationales; `configs/synth_data.yaml`).

### A1 — spotting supervision

Add `value + [x1,y1,x2,y2]` targets (ask_where/region). Hypothesis: grounding lives in vision+connector → adapting the **connector** lifts `cap_ground`/spot-IoU.

In [ ]:
RESULTS = ROOT / "docs" / "results" / "ablation_results.json"

def _load():
    return json.loads(RESULTS.read_text()) if RESULTS.exists() else {"models": {}}

def _summary(runs, key, probe="capability"):
    """Get one probe's summary for an arm, handling the nested {control, probes:{probe:summary}}
    schema written by run_ablation (and the legacy flat schema)."""
    r = runs.get(key)
    if not r:
        return None
    return r.get("probes", {}).get(probe) if "probes" in r else r

def cmd(arm, placement="all"):
    """Print the GPU command that fills this arm (control: --count/--steps held fixed across arms)."""
    print(f"!python scripts/run_ablation.py --arm {arm} --placement {placement} --count 50 --steps 300")

def side_by_side(arm_key, title, axis=None, probe="capability"):
    """Grouped bars: baseline vs <arm> for Qwen3.5. axis=None -> overall score on `probe`;
    axis set -> that by_answer_type cell. Reads ablation_results.json; 'pending' until run on GPU."""
    d = _load().get("models", {})
    fig, ax = plt.subplots(figsize=(7, 4)); w = 0.6; x = np.arange(2); any_data = False
    for i, m in enumerate(MODELS):
        runs = d.get(m, {})
        def score(key):
            s = _summary(runs, key, probe)
            if not s: return None
            return s.get("by_answer_type", {}).get(axis, {}).get("score") if axis else s.get("score")
        b, a = score("baseline"), score(arm_key)
        if b is not None or a is not None: any_data = True
        ax.bar(x + _off(i)*w, [b or 0, a or 0], w, label=m, color=COLOR[m])
    ax.set_xticks(x); ax.set_xticklabels(["baseline", arm_key]); ax.set_ylim(0, 1.05)
    ax.set_title(title + ("" if any_data else "  (run the cell above on GPU to populate)"))
    ax.legend(fontsize=8); plt.tight_layout(); plt.show()

### A1 — spotting supervision

Add `value + [x1,y1,x2,y2]` targets (ask_where/region). Hypothesis: grounding lives in vision+connector → adapting the **connector** lifts `cap_ground`/spot-IoU on both models.

In [ ]:
cmd("A1_spotting_on", "connector")
side_by_side("A1_spotting_on:connector", "A1 — spotting supervision", axis='L1')

### A2 — reasoning supervision

Add `rationale → answer` targets. Hypothesis: relational/numeric reasoning is an **LLM** computation → adapting `llm_attn` (+mlp) closes qwen3.5's H2 gap.

In [ ]:
cmd("A2_reasoning_on", "llm_attn")
side_by_side("A2_reasoning_on:llm_attn", "A2 — reasoning supervision", axis='H2')

### A4 — multilingual mix

Train ko+en (then other pairs). Hypothesis: new vocabulary/script is stored in the **LLM MLP/embeddings**; related scripts transfer, distant interfere at fixed capacity.

In [ ]:
cmd("A4_ko_en", "llm_mlp")
side_by_side("A4_ko_en:llm_mlp", "A4 — multilingual mix", axis=None)

### A5 — LoRA placement

Same A1 data, sweep the placement group {vision|connector|llm_attn|llm_mlp|all}. Hypothesis: grounding gains concentrate in vision+connector, reasoning in the LLM — a capability×module interaction (resolved by introspection, `finetune.lora_vlm.resolve_lora_targets`).

In [ ]:
cmd("A1_spotting_on", "vision")
side_by_side("A1_spotting_on:vision", "A5 — LoRA placement", axis='L1')

### A7 — preprocessing / orientation

Higher-res dynamic tiling + orientation augmentation. Hypothesis: small-text recognition is an encoder+resolution property → `cap_text`/small-text ↑ and 180° rotation retention ↑.

In [ ]:
cmd("A7_dynamic_tiling", "all")
side_by_side("A7_dynamic_tiling:all", "A7 — preprocessing / orientation", axis='T1')

## 4. Cumulative staircase — Qwen3.5

In [ ]:
order = ["baseline", "A7_dynamic_tiling:all", "A1_spotting_on:connector",
         "A2_reasoning_on:llm_attn", "A4_ko_en:llm_mlp"]
d = _load().get("models", {})
fig, ax = plt.subplots(figsize=(10, 4.5))
for m in MODELS:
    runs = d.get(m, {})
    ys = [(_summary(runs, k) or {}).get("score") for k in order]
    xs = [i for i, y in enumerate(ys) if y is not None]
    ax.plot(xs, [ys[i] for i in xs], "-o", label=m, color=COLOR[m]) if xs else None
ax.set_xticks(range(len(order))); ax.set_xticklabels([o.split(":")[0] for o in order], rotation=20, fontsize=8)
ax.set_ylabel("probe score"); ax.set_title("Cumulative ablation staircase (fill via run_ablation.py)")
ax.legend(); plt.tight_layout(); plt.show()
print("Each step stacks the winning arm; a flat/negative step is itself a finding (drop it).")